#### **Q4. Viterbi Algorithm Implementation for the Nature Primer**

#####  1. The HMM(Hidden Markov Model) Parameters :

##### **States :**
**E**: Exon  
**5**: 5′ Splice Site  
**I**: Intron  

##### **Alphabet(Bases) :**
{A, C, G, T}

##### **Transition Probabilities :**
Start → E: 1.0  
E → E: 0.9  
E → 5: 0.1  
5 → I: 1.0  
I → I: 0.9  
I → End: 0.1  

##### **Emission Probabilities :**

| State | A    | C    | G    | T    |
|-------|------|------|------|------|
| **E** | 0.25 | 0.25 | 0.25 | 0.25 |
| **5** | 0.05 | 0.0  | 0.95 | 0.0  |
| **I** | 0.4  | 0.1  | 0.1  | 0.4  |

In [1]:

import numpy as np
# Define states
states = ['E', '5', 'I']

# Transition probabilities
transition_probs = {
    'Start': {'E': 1.0},
    'E': {'E': 0.9, '5': 0.1},
    '5': {'I': 1.0},
    'I': {'I': 0.9, 'End': 0.1}
}

# Emission probabilities
emission_probs = {
    'E': {'A': 0.25, 'C': 0.25, 'G': 0.25, 'T': 0.25},
    '5': {'A': 0.05, 'C': 0.0, 'G': 0.95, 'T': 0.0},
    'I': {'A': 0.4, 'C': 0.1, 'G': 0.1, 'T': 0.4}
}

def get_log_prob_of_a_given_path(state_path, sequence):
    """Calculate log probability of sequence following a given state path"""
    log_prob = 0.0
    prev_state = 'Start'
    
    for state, obs in zip(state_path, sequence):
        trans_prob = transition_probs[prev_state][state]
        emit_prob = emission_probs[state][obs]
        
        # Avoid taking log(0)
        if trans_prob == 0 or emit_prob == 0:
            return -np.inf
        
        log_prob += np.log(trans_prob) + np.log(emit_prob)
        prev_state = state
    
    # Add transition to End
    if prev_state in transition_probs and 'End' in transition_probs[prev_state]:
        log_prob += np.log(transition_probs[prev_state]['End'])
    else:
        return -np.inf
    return log_prob

def viterbi(sequence):
    """Viterbi algorithm to find the most likely state path"""
    n = len(sequence)
    V = [{} for _ in range(n)]   # Dynamic programming table
    path = {}

    # Initialization step
    for state in states:
        if state in transition_probs['Start']:
            trans_prob = transition_probs['Start'][state]
            emit_prob = emission_probs[state][sequence[0]]
            if trans_prob == 0 or emit_prob == 0:
                V[0][state] = -np.inf
            else:
                V[0][state] = np.log(trans_prob) + np.log(emit_prob)
            path[state] = [state]
        else:
            V[0][state] = -np.inf
            path[state] = []

    # Recursion step
    for t in range(1, n):
        new_path = {}
        for curr_state in states:
            max_prob = -np.inf
            best_prev_state = None
            for prev_state in states:
                if prev_state in transition_probs and curr_state in transition_probs[prev_state]:
                    trans_prob = transition_probs[prev_state][curr_state]
                    emit_prob = emission_probs[curr_state][sequence[t]]
                    if trans_prob == 0 or emit_prob == 0:
                        continue
                    prob = V[t-1][prev_state] + np.log(trans_prob) + np.log(emit_prob)
                    if prob > max_prob:
                        max_prob = prob
                        best_prev_state = prev_state
            V[t][curr_state] = max_prob
            if best_prev_state:
                new_path[curr_state] = path[best_prev_state] + [curr_state]
            else:
                new_path[curr_state] = []
        path = new_path

    # Termination step
    max_prob = -np.inf
    best_last_state = None
    for state in states:
        if state in transition_probs and 'End' in transition_probs[state]:
            prob = V[n-1][state] + np.log(transition_probs[state]['End'])
            if prob > max_prob:
                max_prob = prob
                best_last_state = state

    return path[best_last_state], max_prob

sequence = "CTTCATGTGAAAGCAGACGTAAGTCA"

# Given path (from the figure)
state_path = "EEEEEEEEEEEEEEE5IIIIIIIIII"

log_prob_given_path = get_log_prob_of_a_given_path(state_path, sequence)
print(f"Log probability of given path: {log_prob_given_path:.2f}")

# Viterbi most probable path
viterbi_path, viterbi_log_prob = viterbi(sequence)
print(f"Most probable state path: {''.join(viterbi_path)}")
print(f"Log probability of Viterbi path: {viterbi_log_prob:.2f}")


Log probability of given path: -42.58
Most probable state path: EEEEEEEEEEEEEEEEEE5IIIIIII
Log probability of Viterbi path: -41.22
